In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KafkaSparkBase") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1"
    ) \
    .getOrCreate()

spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9ae0a559-af73-4b95-9a25-b894e2d69a42;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.1 in central
	found org.apache.kafka#kafka-clients;3.9.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#

In [2]:
print("Spark:", spark.version)
print("Scala JVM:", spark.sparkContext._jvm.scala.util.Properties.versionNumberString())

Spark: 4.1.1
Scala JVM: 2.13.17


In [11]:
df_kafka = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "ec-kafka:9092") \
    .option("subscribe", "orden-eventos") \
    .option("startingOffsets", "earliest") \
    .load()

In [12]:
df_kafka.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [13]:
df_texto = df_kafka.selectExpr("CAST(value AS STRING) as mensaje")

In [14]:
#.option("checkpointLocation", "/opt/artifacts/checkpoints/ventas_console") \
query_console = df_texto.writeStream \
    .format("console") \
    .outputMode("append") \
    .start()


26/04/16 00:04:06 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-abb60894-c929-4626-b32a-42cd41d38187. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/16 00:04:06 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|                  hi|
|             orden 1|
|{"tipoEvento":"or...|
|{"tipoEvento":"or...|
|{"tipoEvento":"or...|
|{"tipoEvento":"or...|
+--------------------+



In [8]:
#GUARDAR EN PARQUET
query_parquet = df_texto.writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", "/opt/data/ventas_parquet") \
    .option("checkpointLocation", "/opt/artifacts/checkpoints2/ventas_parquet") \
    .start()

26/04/14 10:50:47 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [17]:
!pip install kafka-python
!pip list | grep kafka

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.1/326.1 kB 2.7 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
kafka-python              2.3.1

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [7]:
from kafka import KafkaProducer
import json, time

producer = KafkaProducer(
    bootstrap_servers='kafka-str:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

for i in range(5):
    data = {"mensaje": f"venta {i}"}
    producer.send("ventas", value=data)
    print("Enviado:", data)
    time.sleep(1)

Enviado: {'mensaje': 'venta 0'}


-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"mensaje": "vent...|
+--------------------+

Enviado: {'mensaje': 'venta 1'}


-------------------------------------------
Batch: 2
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"mensaje": "vent...|
+--------------------+

Enviado: {'mensaje': 'venta 2'}
-------------------------------------------
Batch: 3
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"mensaje": "vent...|
+--------------------+

Enviado: {'mensaje': 'venta 3'}
-------------------------------------------
Batch: 4
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"mensaje": "vent...|
+--------------------+

Enviado: {'mensaje': 'venta 4'}


-------------------------------------------
Batch: 5
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"mensaje": "vent...|
+--------------------+

-------------------------------------------
Batch: 6
-------------------------------------------
+--------+
| mensaje|
+--------+
|ventas 8|
+--------+

-------------------------------------------
Batch: 7
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"producto": "Mou...|
+--------------------+

-------------------------------------------
Batch: 8
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"producto": "Lap...|
+--------------------+

-------------------------------------------
Batch: 9
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"producto": "Tec...|
+------------------

-------------------------------------------
Batch: 11
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"producto": "Tec...|
+--------------------+

-------------------------------------------
Batch: 12
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"producto": "Lap...|
+--------------------+

-------------------------------------------
Batch: 13
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"producto": "Mou...|
+--------------------+

-------------------------------------------
Batch: 14
-------------------------------------------
+--------------------+
|             mensaje|
+--------------------+
|{"producto": "Mou...|
+--------------------+

-------------------------------------------
Batch: 15
-------------------------------------------
+--------------------+
|             mensaje|
